# Weeks 5–8 Reflection Code

All of the code behind the Week 5–8 reflections, one section per week. The written answers are in `Kyle_Reynolds_Weeks_5-8_Reflections.pdf`.

Run from the `Module-6-Assignments` folder so the relative `csvs/` paths resolve. Cells run top to bottom.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.spatial.distance import mahalanobis
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors

CSVS = Path("csvs")
SEED = 0

## Week 5 — Causal diagrams and feedback loops

### Q1. Draw a diagram for a negative feedback loop: sweating lowers body temperature, and high body temperature causes sweating

The diagram is in the reflection. A DAG can't have two things cause each other, so the loop is split by time. The simulation follows one runner minute by minute: each minute's temperature sets that minute's sweat (+), and that sweat lowers the next minute's temperature (−). With the sweat arrow turned off, the temperature just keeps climbing.

In [2]:
MINUTES = 20
START = 37.0             # body temperature at the start of the run (°C)
RUN_HEAT = 0.3           # °C the run adds each minute
SWEAT_PER_DEGREE = 0.5   # Temp (t) --(+)--> Sweat (t)
COOL_PER_SWEAT = 0.6     # Sweat (t) --(-)--> Temp (t + 1)


def sweat_loop(sweating=True, seed=SEED):
    """One run, minute by minute. sweating=False removes the Sweat -> Temp arrow."""
    rng = np.random.default_rng(seed)
    temp, rows = START, []
    for minute in range(1, MINUTES + 1):
        sweat = max(SWEAT_PER_DEGREE * (temp - START) + rng.normal(0, 0.05), 0) if sweating else 0.0
        rows.append({"minute": minute, "temp": temp, "sweat": sweat})
        temp = temp + RUN_HEAT - COOL_PER_SWEAT * sweat + rng.normal(0, 0.02)
    return pd.DataFrame(rows)


run = sweat_loop().assign(temp_no_sweat=sweat_loop(sweating=False)["temp"])
print(run.iloc[[0, 1, 2, 4, 9, 14, 19]].round(2).to_string(index=False))

# Levels off where the sweat removes exactly the heat the run adds each minute.
steady = START + RUN_HEAT / (COOL_PER_SWEAT * SWEAT_PER_DEGREE)
print(f"\nwith sweating, temperature levels off near {steady:.1f} °C")

 minute  temp  sweat  temp_no_sweat
      1 37.00   0.01          37.00
      2 37.29   0.18          37.30
      3 37.49   0.22          37.60
      5 37.75   0.34          38.21
     10 38.02   0.53          39.74
     15 38.02   0.48          41.15
     20 38.04   0.56          42.61

with sweating, temperature levels off near 38.0 °C


### Q2. Describe an example of a positive feedback loop

Google reviews: more reviews push a restaurant up the search results, which brings in more customers (+), and more customers leave more reviews (+). Each week's reviews set that week's customers, and their reviews add to next week's total, so the growth speeds up instead of settling.

In [3]:
WEEKS = 10
START_REVIEWS = 10
CUSTOMERS_PER_REVIEW = 2   # Reviews (week) --(+)--> Customers (week): higher in search
REVIEW_RATE = 0.1          # Customers (week) --(+)--> Reviews (next week): 1 in 10 leaves one


def reviews_loop(seed=SEED):
    rng = np.random.default_rng(seed)
    reviews, rows = START_REVIEWS, []
    for week in range(1, WEEKS + 1):
        customers = rng.poisson(20 + CUSTOMERS_PER_REVIEW * reviews)
        new_reviews = rng.binomial(customers, REVIEW_RATE)
        rows.append({"week": week, "reviews": reviews, "customers": customers,
                     "new reviews": new_reviews})
        reviews += new_reviews
    return pd.DataFrame(rows)


print(reviews_loop().to_string(index=False))

 week  reviews  customers  new reviews
    1       10         43            1
    2       11         53            7
    3       18         57            8
    4       26         63            6
    5       32         78            3
    6       35         77            9
    7       44        111           21
    8       65        155           14
    9       79        161           16
   10       95        202           25


### Q3. Simulate lightning, bears, deer, and flowers, and find a backdoor path between deer and flowers

One row is one forest region in one season, and every node gets its own noise. Deer ← Lightning → Flowers and Deer ← Bears ← Lightning → Flowers are backdoor paths, so the coefficient of deer only lands on the true −3 once lightning is controlled for. Bears adds nothing once lightning is in: its only way to flowers besides deer is through lightning.

In [4]:
N_REGIONS = 10_000
DEER_ON_FLOWERS = -3.0    # true effect: each deer eats 3 flowers


def simulate_forest(n=N_REGIONS, seed=SEED):
    rng = np.random.default_rng(seed)
    lightning = np.maximum(rng.poisson(6, n) + rng.normal(0, 0.5, n), 0)   # storms per season
    bears = np.maximum(20 - 1.0 * lightning + rng.normal(0, 3, n), 0)
    deer = np.maximum(100 - 4.0 * lightning - 1.5 * bears + rng.normal(0, 8, n), 0)
    flowers = np.maximum(
        600 + 25.0 * lightning + DEER_ON_FLOWERS * deer + rng.normal(0, 30, n), 0)
    return pd.DataFrame({"lightning": lightning, "bears": bears,
                         "deer": deer, "flowers": flowers})


def coef_of_deer(df, controls=()):
    X = sm.add_constant(df[["deer", *controls]])
    return sm.OLS(df["flowers"], X).fit().params["deer"]


forest = simulate_forest()
print(forest.describe().loc[["mean", "std", "min", "max"]].round(2))
print("\ncorrelations:")
print(forest.corr().round(3))

print(f"\n{'flowers ~':<30} {'coef of deer':>13}")
for controls in [(), ("bears",), ("lightning",), ("lightning", "bears")]:
    label = " + ".join(["deer", *controls])
    print(f"{label:<30} {coef_of_deer(forest, controls):13.4f}")
print(f"{'true effect':<30} {DEER_ON_FLOWERS:13.4f}")

      lightning  bears   deer  flowers
mean       5.99  14.01  54.90   585.08
std        2.49   3.90  11.09    90.96
min        0.00   0.00   9.09   292.88
max       17.07  28.82  94.65   981.41

correlations:
           lightning  bears   deer  flowers
lightning      1.000 -0.635 -0.559    0.893
bears         -0.635  1.000  0.033   -0.448
deer          -0.559  0.033  1.000   -0.754
flowers        0.893 -0.448 -0.754    1.000

flowers ~                       coef of deer
deer                                 -6.1814
deer + bears                         -6.0653
deer + lightning                     -3.0368
deer + lightning + bears             -3.0391
true effect                          -3.0000


### Q4. Draw a diagram of your own with at least four nodes, one confounder, and one collider

Treatment (X) is going to the gym and outcome (Y) is losing weight. Motivation is the confounder (Gym ← Motivation → Eating better → Losing weight), and buying new clothes is the collider (Gym → Buying new clothes ← Losing weight). The simulation sets the true effect at 1 lb per weekly gym hour, then fits weight loss on gym three ways: alone, controlling for motivation, and controlling for motivation but looking only at people who bought new clothes.

In [5]:
N_PEOPLE = 10_000
GYM_ON_LOSS = 1.0        # true effect: 1 lb lost per weekly gym hour


def simulate_gym(n=N_PEOPLE, seed=SEED):
    """Motivation -> gym and eating; gym and eating -> weight loss;
    gym and weight loss -> buying new clothes."""
    rng = np.random.default_rng(seed)
    motivation = rng.normal(0, 1, n)
    gym = np.maximum(3 + 1.0 * motivation + rng.normal(0, 1, n), 0)        # hours a week
    eating = 1.0 * motivation + rng.normal(0, 1, n)                         # how well they eat
    weight_loss = GYM_ON_LOSS * gym + 2.0 * eating + rng.normal(0, 1, n)    # lbs
    clothes = (gym + weight_loss + rng.normal(0, 1, n) > 6).astype(int)     # 1 = bought new clothes
    return pd.DataFrame({"motivation": motivation, "gym": gym, "eating": eating,
                         "weight_loss": weight_loss, "clothes": clothes})


people = simulate_gym()
buyers = people[people["clothes"] == 1]
print(f"bought new clothes: {len(buyers)} of {len(people)}\n")

print(f"{'weight_loss ~':<32} {'rows':>6} {'coef of gym':>12}")
for label, formula, df in [
    ("gym", "weight_loss ~ gym", people),                                  # backdoor open
    ("gym + motivation", "weight_loss ~ gym + motivation", people),        # backdoor closed
    ("gym + motivation, buyers only", "weight_loss ~ gym + motivation", buyers),  # collider opened
]:
    print(f"{label:<32} {len(df):>6} {smf.ols(formula, df).fit().params['gym']:12.4f}")
print(f"{'true effect':<32} {'':>6} {GYM_ON_LOSS:12.4f}")

bought new clothes: 5029 of 10000

weight_loss ~                      rows  coef of gym
gym                               10000       2.0193
gym + motivation                  10000       0.9941
gym + motivation, buyers only      5029       0.5071
true effect                                   1.0000


## Week 6 — The optimal treatment effect (`homework_6.1.csv`)

### Q1. What is the problem with taking the maximum difference between each untreated item and its counterfactual?

Match every row on Z to its nearest row in the other group; that match is the row's counterfactual. The quiz's answer is the largest effect among the untreated rows. The quiz data has no noise, so that max is right. To see the problem, simulate the same setup (true effect = 1.2 + Z) with noise added to Y, and compare the max with the true optimal effect over 200 datasets.

In [6]:
def counterfactual_y(rows, other_group):
    """Y of the nearest row (by Z) in the other group, one per row."""
    nn = NearestNeighbors(n_neighbors=1).fit(other_group[["Z"]])
    _, indices = nn.kneighbors(rows[["Z"]])
    return other_group["Y"].to_numpy()[indices.ravel()]


def unit_effects(df):
    """Treated minus untreated for every row: (effects on treated, effects on untreated)."""
    treated = df[df["X"] == 1]
    untreated = df[df["X"] == 0]
    # Treated row: its own Y is the treated outcome, the match is what it would be untreated.
    on_treated = treated["Y"].to_numpy() - counterfactual_y(treated, untreated)
    # Untreated row: the match is what it would be treated, its own Y is the untreated outcome.
    on_untreated = counterfactual_y(untreated, treated) - untreated["Y"].to_numpy()
    return on_treated, on_untreated


quiz = pd.read_csv(CSVS / "homework_6.1.csv")
on_treated, on_untreated = unit_effects(quiz)
print(f"{len(on_treated)} treated, {len(on_untreated)} untreated")
print(f"max untreated effect: {on_untreated.max():.4f}")
print(f"true optimal effect:  {1.2 + quiz.loc[quiz['X'] == 0, 'Z'].max():.4f}  (1.2 + largest untreated Z)")

491 treated, 509 untreated
max untreated effect: 2.1725
true optimal effect:  2.1729  (1.2 + largest untreated Z)


In [7]:
N_ROWS = 1_000
N_EXPERIMENTS = 200
NOISE_LEVELS = [0.0, 0.1, 0.3]


def simulate_noisy(noise, rng, n=N_ROWS):
    """Same shape as homework_6.1.csv, plus noise on Y. True effect = 1.2 + Z."""
    z = rng.uniform(0, 1, n)
    x = (rng.uniform(0, 1, n) < z).astype(int)   # higher Z -> more likely treated
    y = np.where(x == 1, 1.2 - 0.5 * z, -1.5 * z) + rng.normal(0, noise, n)
    return pd.DataFrame({"Z": z, "X": x, "Y": y})


def max_error(df):
    """Max untreated effect minus the true optimal effect."""
    _, on_untreated = unit_effects(df)
    return on_untreated.max() - (1.2 + df.loc[df["X"] == 0, "Z"].max())


rng = np.random.default_rng(SEED)
print(f"{N_EXPERIMENTS} datasets per noise level -- max minus true optimal:")
print(f"{'noise':>6} {'mean error':>11} {'sd':>7} {'lowest':>8}")
for noise in NOISE_LEVELS:
    errors = np.array([max_error(simulate_noisy(noise, rng)) for _ in range(N_EXPERIMENTS)])
    print(f"{noise:6.1f} {errors.mean():+11.3f} {errors.std(ddof=1):7.3f} {errors.min():+8.3f}")

200 datasets per noise level -- max minus true optimal:
 noise  mean error      sd   lowest
   0.0      -0.000   0.000   -0.001
   0.1      +0.196   0.078   +0.013
   0.3      +0.854   0.179   +0.472


### Q2. Propose a solution that remedies this problem and implement it: the smoothed max

Fit a line of effect on Z through all of the untreated rows and take the highest point on that line, so one lucky row barely moves the answer. The simulation reuses the same seed, so these are the same 200 datasets per noise level as above, now comparing the max, the 90th percentile, and the smoothed max.

In [8]:
def estimates(df):
    """(true optimal, max, 90th pct, smoothed max) for one dataset."""
    _, on_untreated = unit_effects(df)
    untreated = df[df["X"] == 0].assign(effect=on_untreated)

    true_optimal = 1.2 + untreated["Z"].max()
    line = smf.ols("effect ~ Z", untreated).fit()
    smoothed_max = line.predict(untreated).max()
    return true_optimal, on_untreated.max(), np.percentile(on_untreated, 90), smoothed_max


print("homework_6.1.csv (no noise):")
truth, raw_max, p90, smoothed = estimates(quiz)
print(f"  true {truth:.4f}   max {raw_max:.4f}   90th pct {p90:.4f}   smoothed max {smoothed:.4f}")

rng = np.random.default_rng(SEED)
print(f"\nSimulated, {N_EXPERIMENTS} datasets per noise level -- mean error vs. true optimal (sd):")
print(f"{'noise':>6} {'max':>16} {'90th pct':>16} {'smoothed max':>16}")
for noise in NOISE_LEVELS:
    runs = np.array([estimates(simulate_noisy(noise, rng)) for _ in range(N_EXPERIMENTS)])
    errors = runs[:, 1:] - runs[:, [0]]
    cells = [f"{m:+.3f} ({s:.3f})" for m, s in zip(errors.mean(0), errors.std(0, ddof=1))]
    print(f"{noise:6.1f} " + " ".join(f"{c:>16}" for c in cells))

homework_6.1.csv (no noise):
  true 2.1729   max 2.1725   90th pct 1.9280   smoothed max 2.1734

Simulated, 200 datasets per noise level -- mean error vs. true optimal (sd):
 noise              max         90th pct     smoothed max
   0.0   -0.000 (0.000)   -0.278 (0.027)   +0.002 (0.003)
   0.1   +0.196 (0.078)   -0.252 (0.031)   -0.003 (0.037)
   0.3   +0.854 (0.179)   -0.007 (0.051)   +0.005 (0.107)


## Week 7 — Omitted confounders and repeated tests

### Q1. Leave a confounder out of a linear regression: is the effect of X overestimated, underestimated, or neither?

X = hours of tutoring, Y = test score, W = motivation. Motivation raises tutoring, and each hour of tutoring adds 1 point. Fit Y ~ X (W left out) and Y ~ X + W, once with motivation adding 2 points and once with it taking 2 away.

In [9]:
N = 1_000
rng = np.random.default_rng(SEED)   # Q2 keeps drawing from this same generator


def confounder(w_on_y):
    """Coef of X with W left out vs. with W controlled."""
    w = rng.normal(0, 1, N)                       # motivation
    x = w + rng.normal(0, 1, N)                   # tutoring
    y = 1.0 * x + w_on_y * w + rng.normal(0, 1, N)   # test score
    df = pd.DataFrame({"W": w, "X": x, "Y": y})
    left_out = smf.ols("Y ~ X", df).fit().params["X"]
    controlled = smf.ols("Y ~ X + W", df).fit().params["X"]
    return left_out, controlled


print("true effect of X = 1.0")
for w_on_y in [2.0, -2.0]:
    left_out, controlled = confounder(w_on_y)
    print(f"  W -> Y = {w_on_y:+.0f}:  Y ~ X {left_out:.4f}   Y ~ X + W {controlled:.4f}")

true effect of X = 1.0
  W -> Y = +2:  Y ~ X 1.9351   Y ~ X + W 1.0074
  W -> Y = -2:  Y ~ X -0.0759   Y ~ X + W 0.9534


### Q2. Run a regression where one coefficient is truly zero 1,000 times and report the smallest p-value

W has no effect on Y. Fit Y ~ X + W on 1,000 fresh datasets and keep W's p-value each time.

In [10]:
def p_value_of_w():
    w = rng.normal(0, 1, N)
    x = rng.normal(0, 1, N)
    y = 2 * x + rng.normal(0, 1, N)     # W is not in here
    df = pd.DataFrame({"W": w, "X": x, "Y": y})
    return smf.ols("Y ~ X + W", df).fit().pvalues["W"]


p = np.array([p_value_of_w() for _ in range(N)])
print(f"smallest p-value of W: {p.min():.6f}")
print(f"runs with p < 0.05:    {(p < 0.05).sum()} of {N}  (chance alone expects about {0.05 * N:.0f})")

smallest p-value of W: 0.002175
runs with p < 0.05:    42 of 1000  (chance alone expects about 50)


## Week 8 — Inverse probability weighting and Mahalanobis matching

### Questions 1 and 2. Inverse probability weighting (`homework_8.1.csv`)

A logistic regression of X on Z gives each row's propensity score, P(X = 1 | Z). Treated rows get weight 1/P and untreated rows 1/(1 − P), and the ATE is the weighted mean of Y for the treated minus the weighted mean for the untreated.

In [11]:
def propensity_scores(df):
    """P(X = 1 | Z) from a logistic regression of X on Z."""
    model = LogisticRegression().fit(df[["Z"]], df["X"])
    # predict_proba gives one column per class, [P(X = 0), P(X = 1)].
    return model.predict_proba(df[["Z"]])[:, 1]


def ipw_ate(df, p):
    """Weighted mean of Y for treated minus weighted mean of Y for untreated."""
    treated = df["X"] == 1
    weights = np.where(treated, 1 / p, 1 / (1 - p))
    y_treated = np.average(df["Y"][treated], weights=weights[treated])
    y_untreated = np.average(df["Y"][~treated], weights=weights[~treated])
    return y_treated - y_untreated


ipw = pd.read_csv(CSVS / "homework_8.1.csv")
p = propensity_scores(ipw)
print(f"Q1 IPW ATE: {ipw_ate(ipw, p):.4f}  -> B (2.3)")
print(f"Q2 first three propensity scores: {np.round(p[:3], 2)}  -> A [0.84, 0.58, 0.71]")

Q1 IPW ATE: 2.2743  -> B (2.3)
Q2 first three propensity scores: [0.84 0.58 0.71]  -> A [0.84, 0.58, 0.71]


### Questions 3 and 4. Mahalanobis matching (`homework_8.2.csv`)

Each treated row is matched, with replacement, to the untreated row closest to it in Mahalanobis distance on (Z1, Z2). The treated row with the least common support is the one whose best match is still the farthest away, and Q4 asks for the values of that match.

In [12]:
def inverse_covariance(df):
    """Invert the 2x2 covariance of all Z1 and Z2 values (a 2xN matrix)."""
    z = df[["Z1", "Z2"]].to_numpy().T
    return np.linalg.inv(np.cov(z))


def match_treated(df):
    """For each treated row: its Y minus its nearest untreated Y, the distance to
    that match, and the match's row.

    With replacement -- every treated row searches all untreated rows, so the
    same untreated row can be the match more than once.
    """
    vi = inverse_covariance(df)
    treated = df[df["X"] == 1]
    untreated = df[df["X"] == 0]
    untreated_z = untreated[["Z1", "Z2"]].to_numpy()

    effects, distances, matches = [], [], []
    for z, y in zip(treated[["Z1", "Z2"]].to_numpy(), treated["Y"]):
        d = np.array([mahalanobis(z, u, vi) for u in untreated_z])
        nearest = d.argmin()
        effects.append(y - untreated["Y"].iloc[nearest])
        distances.append(d[nearest])
        matches.append(untreated.iloc[nearest])
    return treated, np.array(effects), np.array(distances), matches


matching = pd.read_csv(CSVS / "homework_8.2.csv")
treated, effects, distances, matches = match_treated(matching)
print(f"Q3 Mahalanobis-matched ATE: {effects.mean():.4f}  -> D (3.4)")

# Least common support = the treated row whose best match is still the farthest away.
# Q4 asks for that row's nearest match, not the treated row itself.
worst = distances.argmax()
row, match = treated.iloc[worst], matches[worst]
print(f"Q4 least common support: treated row at Z1 = {row['Z1']:.4f}, Z2 = {row['Z2']:.4f}")
print(f"   its nearest untreated match: Z1 = {match['Z1']:.4f}, Z2 = {match['Z2']:.4f}, "
      f"distance {distances[worst]:.4f}  -> D (1.5, -1.3)")

Q3 Mahalanobis-matched ATE: 3.4377  -> D (3.4)
Q4 least common support: treated row at Z1 = 2.6962, Z2 = 0.5382
   its nearest untreated match: Z1 = 1.5200, Z2 = -1.2822, distance 1.3830  -> D (1.5, -1.3)


### Obstacles, challenges and insights

The checks behind section 3 of the reflection: why the covariance needs `.T`, how plain (Euclidean) distance compares with Mahalanobis, and how far the worst match is next to a typical one.

In [13]:
z = matching[["Z1", "Z2"]].to_numpy()          # N x 2: one row per item
print(f"np.cov(z).shape   = {np.cov(z).shape}   (compares rows with each other)")
print(f"np.cov(z.T).shape = {np.cov(z.T).shape}       (compares Z1 with Z2)")
print(f"\nspread of each Z: sd Z1 = {z[:, 0].std(ddof=1):.3f}, sd Z2 = {z[:, 1].std(ddof=1):.3f}, "
      f"corr = {np.corrcoef(z.T)[0, 1]:.3f}")

# Plain distance: nearest untreated row by straight-line distance on (Z1, Z2).
untreated = matching[matching["X"] == 0]
nn = NearestNeighbors(n_neighbors=1).fit(untreated[["Z1", "Z2"]])
_, indices = nn.kneighbors(treated[["Z1", "Z2"]])
euclidean = treated["Y"].to_numpy() - untreated["Y"].to_numpy()[indices.ravel()]
print(f"\nmatched ATE, Mahalanobis: {effects.mean():.4f}")
print(f"matched ATE, Euclidean:   {euclidean.mean():.4f}")

# Common support: the worst match against a typical one.
print(f"\nMahalanobis match distance: median {np.median(distances):.3f}, "
      f"90th pct {np.percentile(distances, 90):.3f}, farthest {distances.max():.3f}")

np.cov(z).shape   = (1000, 1000)   (compares rows with each other)
np.cov(z.T).shape = (2, 2)       (compares Z1 with Z2)

spread of each Z: sd Z1 = 0.988, sd Z2 = 1.361, corr = 0.703

matched ATE, Mahalanobis: 3.4377
matched ATE, Euclidean:   3.4779

Mahalanobis match distance: median 0.110, 90th pct 0.532, farthest 1.383
